## Data Source

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("retailrocket/ecommerce-dataset")

print("Path to dataset files:", path)

C:\Users\smako\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\smako\.cache\kagglehub\datasets\retailrocket\ecommerce-dataset\versions\2


# Complementary Product Recommendation System

In [3]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

In [4]:
events = pd.read_csv("data/events.csv")
events.head(10)

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
5,1433224086234,972639,view,22556,NaN
6,1433221923240,810725,view,443030,NaN
7,1433223291897,794181,view,439202,NaN
8,1433220899221,824915,view,428805,NaN
9,1433221204592,339335,view,82389,NaN


In [5]:
events.info()
print(events.columns)
print(events["event"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB
Index(['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'], dtype='str')
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


## Sessionization

In [6]:
add_to_cart = events[events["event"] == "addtocart"].sort_values(["visitorid", "timestamp"]).copy()
add_to_cart

,timestamp,visitorid,event,itemid,transactionid
1133190,1440914628202,6,addtocart,65273,NaN
138668,1433719818230,150,addtocart,452955,NaN
1361435,1441951932678,155,addtocart,368372,NaN
1357916,1441951937583,155,addtocart,452082,NaN
1368056,1441951967871,155,addtocart,181405,NaN
...,...,...,...,...,...
2024081,1432953588975,1407512,addtocart,290723,NaN
2023974,1432954196910,1407512,addtocart,398091,NaN
2009117,1432954247297,1407512,addtocart,241408,NaN
2022592,1432954281393,1407512,addtocart,398533,NaN


In [7]:
SESSION_GAP_MS = 1000000

In [8]:
add_to_cart["time"] = pd.to_datetime(
    add_to_cart["timestamp"],
    unit="ms"
) # Перетворення часу

add_to_cart["gap_ms"] = (
    add_to_cart.groupby("visitorid")["timestamp"]
    .diff()
) # Різниця з попередньою подією

add_to_cart["is_new_session"] = (
    add_to_cart["gap_ms"].isna()
    | (add_to_cart["gap_ms"] > SESSION_GAP_MS)
) # Позначення початку сесії

add_to_cart["session_number"] = (
    add_to_cart.groupby("visitorid")["is_new_session"]
    .cumsum()
) # Нумерація сесій

add_to_cart["session_id"] = (
    add_to_cart["visitorid"].astype(str)
    + "_"
    + add_to_cart["session_number"].astype(str)
) # Створення ідентифікатора сесії

In [9]:
add_to_cart.head(50)


,timestamp,visitorid,event,itemid,transactionid,time,gap_ms,is_new_session,session_number,session_id
1133190,1440914628202,6,addtocart,65273,NaN,2015-08-30 06:03:48.202,NaN,True,1,6_1
138668,1433719818230,150,addtocart,452955,NaN,2015-06-07 23:30:18.230,NaN,True,1,150_1
1361435,1441951932678,155,addtocart,368372,NaN,2015-09-11 06:12:12.678,NaN,True,1,155_1
1357916,1441951937583,155,addtocart,452082,NaN,2015-09-11 06:12:17.583,4905.0,False,1,155_1
1368056,1441951967871,155,addtocart,181405,NaN,2015-09-11 06:12:47.871,30288.0,False,1,155_1
1367869,1441952264613,155,addtocart,41882,NaN,2015-09-11 06:17:44.613,296742.0,False,1,155_1
1367862,1441952265950,155,addtocart,442601,NaN,2015-09-11 06:17:45.950,1337.0,False,1,155_1
1364941,1441952340395,155,addtocart,224623,NaN,2015-09-11 06:19:00.395,74445.0,False,1,155_1
1366578,1442032680026,155,addtocart,389974,NaN,2015-09-12 04:38:00.026,80339631.0,True,2,155_2
848067,1439599816912,172,addtocart,10034,NaN,2015-08-15 00:50:16.912,NaN,True,1,172_1


In [10]:
session_items = (
    add_to_cart.groupby("session_id")["itemid"]
    .agg(list)
    .reset_index()
)

session_items.head(10)

,session_id,itemid
0,1000057_1,[271494]
1,100007_1,"[349736, 349736]"
2,1000093_1,[199101]
3,1000093_2,[199101]
4,1000103_1,[440560]
5,100012_1,[196659]
6,1000169_1,[218942]
7,100020_1,[70387]
8,1000248_1,[37471]
9,1000306_1,[170390]


## list of recommendations for all items

In [11]:
recommend_pairs = []

for items in session_items["itemid"]:
    for pair in combinations(set(items), 2):
        recommend_pairs.append(tuple(sorted(pair)))

In [12]:
pair_counts = pd.Series(recommend_pairs).value_counts()
pair_counts.head()

(218794, 461686)    25
(171878, 461686)    23
(10572, 461686)     19
(32581, 461686)     17
(208939, 444190)    14
Name: count, dtype: int64

In [13]:
MIN_PAIR_COUNT = 3

frequent_pairs = pair_counts[pair_counts >= MIN_PAIR_COUNT]
frequent_pairs.head()

(218794, 461686)    25
(171878, 461686)    23
(10572, 461686)     19
(32581, 461686)     17
(208939, 444190)    14
Name: count, dtype: int64

In [14]:
len(pair_counts), len(frequent_pairs)

(49378, 548)

In [15]:
recommendations = defaultdict(list)

for pair, count in frequent_pairs.items():
    item_a, item_b = pair

    recommendations[item_a].append((item_b, count))
    recommendations[item_b].append((item_a, count))

for item_id in recommendations:
    recommendations[item_id].sort(
        key=lambda recommendation: recommendation[1],
        reverse=True
    )

### checking on exact item

In [16]:
some_item_id = next(iter(recommendations))
print(some_item_id)
print(recommendations[some_item_id])

218794
[(461686, 25), (171878, 8), (32581, 5), (10572, 4), (431632, 3)]


In [17]:
def get_recommendations(item_id, top_n=3):
    if item_id not in recommendations:
        return []

    return recommendations[item_id][:top_n]

In [18]:
get_recommendations(some_item_id)

[(461686, 25), (171878, 8), (32581, 5)]

### Getting categories for items

In [19]:
properties_part1 = pd.read_csv("data/item_properties_part1.csv")
properties_part1.head(15)

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513
5,1436065200000,285026,available,0
6,1434250800000,89534,213,1121373
7,1431831600000,264312,6,319724
8,1433646000000,229370,202,1330310
9,1434250800000,98113,451,1141052 n48.000


In [20]:
print(properties_part1["property"].value_counts().head(20))

property
888           1629817
790            970800
available      817387
categoryid     426305
6              343207
283            323681
776            311654
678            261829
364            256340
202            242984
839            226921
159            226502
917            226437
764            226242
112            226102
227            188209
698            157281
451            142388
663            131331
962            128976
Name: count, dtype: int64


In [21]:
category_rows1 = properties_part1[
    properties_part1["property"] == "categoryid"
]

category_rows1.head(10)

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
140,1432436400000,281245,categoryid,1277
151,1435460400000,35575,categoryid,1059
189,1437274800000,8313,categoryid,1147
197,1437879600000,55102,categoryid,47
213,1433041200000,397079,categoryid,619
237,1436670000000,265036,categoryid,1228
254,1437879600000,124459,categoryid,1277
310,1437879600000,350508,categoryid,546
325,1439089200000,221365,categoryid,1226


In [22]:
category_rows1[category_rows1["itemid"] == some_item_id]

,timestamp,itemid,property,value


In [23]:
properties_part1[
    properties_part1["itemid"] == some_item_id
].head(10)

,timestamp,itemid,property,value
169616,1433041200000,218794,28,150169 610517
894804,1439694000000,218794,550,769062
3350068,1432436400000,218794,810,1097825 421694 n12.000 n60.000 1175087 n264.00...
3385029,1433646000000,218794,558,550504 827388
4082042,1433041200000,218794,364,46664
4241961,1431226800000,218794,917,n5688.000
4379147,1431226800000,218794,790,n16620.000
4577931,1431831600000,218794,790,n16620.000
4776715,1433646000000,218794,790,n14640.000
4975499,1432436400000,218794,790,n16620.000


#### read item_properties_part2

In [24]:
properties_part2 = pd.read_csv("data/item_properties_part2.csv")

In [25]:
properties_part2[
    (properties_part2["itemid"] == some_item_id)
    & (properties_part2["property"] == "categoryid")
]

,timestamp,itemid,property,value
7352774,1431226800000,218794,categoryid,1037


### Merging category rows

In [26]:
all_category_rows = pd.concat([
    properties_part1[properties_part1["property"] == "categoryid"],
    properties_part2[properties_part2["property"] == "categoryid"]
])

all_category_rows.head(10)

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
140,1432436400000,281245,categoryid,1277
151,1435460400000,35575,categoryid,1059
189,1437274800000,8313,categoryid,1147
197,1437879600000,55102,categoryid,47
213,1433041200000,397079,categoryid,619
237,1436670000000,265036,categoryid,1228
254,1437879600000,124459,categoryid,1277
310,1437879600000,350508,categoryid,546
325,1439089200000,221365,categoryid,1226


## list: itemid - categoryid

In [27]:
latest_categories = all_category_rows.sort_values("timestamp").drop_duplicates(subset="itemid", keep="last")[["itemid", "value"]].rename(columns={"value": "categoryid"})

latest_categories.head(10)

,itemid,categoryid
9273909,419938,1680
9273753,321873,959
2126,388926,374
2986484,336336,598
1788,197821,342
1699,10316,342
1674,30822,769
1441,38175,196
1276,152242,173
1176,321776,1196


## Percentage of items that have categories

In [28]:
recommended_item_ids = set(recommendations.keys())
categorized_item_ids = set(latest_categories["itemid"])

coverage = len(recommended_item_ids & categorized_item_ids) / len(recommended_item_ids)

print(f"Товарів із категорією: {coverage:.1%}")

Товарів із категорією: 100.0%


In [30]:
counts_add_to_cart_events = add_to_cart["itemid"].value_counts()
counts_add_to_cart_events.head(10)

itemid
461686    306
312728    162
409804    156
320130    141
29196     136
7943       97
48030      95
441852     90
445351     89
257040     89
Name: count, dtype: int64

In [31]:
item_popularity = (
    add_to_cart["itemid"]
    .value_counts()
    .rename_axis("itemid")
    .reset_index(name="addtocart_count")
)

item_popularity.head()

,itemid,addtocart_count
0,461686,306
1,312728,162
2,409804,156
3,320130,141
4,29196,136


In [46]:
category_popularity = pd.merge(
    item_popularity,
    latest_categories,
    on="itemid",
    how="left"
)

category_popularity = category_popularity.dropna(
    subset=["categoryid"]
).copy()

category_popularity.head(10)

,itemid,addtocart_count,categoryid
0,461686,306,1037
1,312728,162,1098
2,409804,156,1191
3,320130,141,1483
4,29196,136,1265
5,7943,97,398
6,48030,95,1219
7,441852,90,959
8,445351,89,1483
9,257040,89,683


## Category popularity

In [47]:
category_popularity = category_popularity.sort_values(
    ["categoryid", "addtocart_count"],
    ascending=[True, False]
)

items_by_category = (
    category_popularity
    .groupby("categoryid")["itemid"]
    .agg(list)
    .to_dict()
)

In [48]:
item_to_category = (
    category_popularity
    .set_index("itemid")["categoryid"]
    .to_dict()
)

In [49]:
def get_similar_items(item_id, top_n=5):
    category_id = item_to_category.get(item_id)

    if category_id is None:
        return []

    category_items = items_by_category[category_id]

    return [
        candidate_id
        for candidate_id in category_items
        if candidate_id != item_id
    ][:top_n]

In [50]:
test_item_id = category_popularity.iloc[0]["itemid"]

print(test_item_id)
print(get_similar_items(test_item_id))

101156
[283492, 289006, 380388, 220950, 132853]


In [51]:
test_category = item_to_category[test_item_id]

for candidate_id in get_similar_items(test_item_id):
    print(
        candidate_id,
        item_to_category[candidate_id] == test_category
    )

283492 True
289006 True
380388 True
220950 True
132853 True


## Data Quality Checks

In [53]:
items_per_session = (
    add_to_cart
    .groupby("session_id")["itemid"]
    .nunique()
)

average_items_per_session = items_per_session.mean()

print(average_items_per_session)

items_per_session.describe()

1.3473560220436667


count    47542.000000
mean         1.347356
std          1.325949
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         91.000000
Name: itemid, dtype: float64

In [54]:
multi_item_sessions = items_per_session[items_per_session >= 2]

print(f"Multi-item sessions: {len(multi_item_sessions)}")
print(f"Share: {len(multi_item_sessions) / len(items_per_session):.1%}")

Multi-item sessions: 7954
Share: 16.7%


In [55]:
multi_item_sessions.quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])

0.50     2.0
0.75     3.0
0.90     5.0
0.95     7.0
0.99    13.0
1.00    91.0
Name: itemid, dtype: float64

In [56]:
pairs_per_session = (
    items_per_session
    * (items_per_session - 1)
    // 2
)

for threshold in [5, 10, 20, 30]:
    large_sessions = items_per_session > threshold

    session_share = large_sessions.mean()
    pair_share = (
        pairs_per_session[large_sessions].sum()
        / pairs_per_session.sum()
    )

    print(
        f">{threshold} items: "
        f"{session_share:.2%} of sessions, "
        f"{pair_share:.2%} of pairs"
    )

>5 items: 1.36% of sessions, 68.23% of pairs
>10 items: 0.29% of sessions, 46.41% of pairs
>20 items: 0.05% of sessions, 27.29% of pairs
>30 items: 0.02% of sessions, 20.00% of pairs


## Large Session Filtering

Sessions with more than 20 unique items were excluded.

Although they represented only 0.05% of sessions, they generated 27.05% of all product pairs.
The filter reduces the influence of unusually large sessions on co-occurrence recommendations.

In [57]:
MAX_SESSION_ITEMS = 20

valid_session_ids = items_per_session[
    items_per_session <= MAX_SESSION_ITEMS
].index

filtered_add_to_cart = add_to_cart[
    add_to_cart["session_id"].isin(valid_session_ids)
].copy()

In [58]:
filtered_session_items = (
    filtered_add_to_cart
    .groupby("session_id")["itemid"]
    .agg(list)
    .reset_index()
)

In [59]:
filtered_recommend_pairs = []

for items in filtered_session_items["itemid"]:
    for pair in combinations(set(items), 2):
        filtered_recommend_pairs.append(tuple(sorted(pair)))

In [60]:
filtered_pair_counts = pd.Series(
    filtered_recommend_pairs
).value_counts()

filtered_frequent_pairs = filtered_pair_counts[
    filtered_pair_counts >= 3
]

In [61]:
print("Original pairs:", len(recommend_pairs))
print("Filtered pairs:", len(filtered_recommend_pairs))

print("Original frequent pairs:", len(frequent_pairs))
print("Filtered frequent pairs:", len(filtered_frequent_pairs))

Original pairs: 52917
Filtered pairs: 38477
Original frequent pairs: 548
Filtered frequent pairs: 464


In [62]:
filtered_recommendations = defaultdict(list)

for pair, count in filtered_frequent_pairs.items():
    item_a, item_b = pair

    filtered_recommendations[item_a].append((item_b, count))
    filtered_recommendations[item_b].append((item_a, count))

for item_id in filtered_recommendations:
    filtered_recommendations[item_id].sort(
        key=lambda recommendation: recommendation[1],
        reverse=True
    )

In [63]:
common_item_ids = (
    set(recommendations)
    & set(filtered_recommendations)
)

test_item_id = next(iter(common_item_ids))

print("Item:", test_item_id)
print("Original:", recommendations[test_item_id][:5])
print("Filtered:", filtered_recommendations[test_item_id][:5])

Item: 274435
Original: [(369447, 4)]
Filtered: [(369447, 4)]


In [64]:
changed_item_ids = [
    item_id
    for item_id in common_item_ids
    if recommendations[item_id][:5]
    != filtered_recommendations[item_id][:5]
]

print("Items with changed top-5:", len(changed_item_ids))

Items with changed top-5: 7


In [65]:
test_item_id = changed_item_ids[0]

print("Item:", test_item_id)
print("Original:", recommendations[test_item_id][:5])
print("Filtered:", filtered_recommendations[test_item_id][:5])

Item: 38965
Original: [(325310, 3), (319680, 3)]
Filtered: [(319680, 3)]


## Related Product Recommender

The final baseline combines complementary recommendations from filtered cart co-occurrence data
with similar items from the same category.

The system prioritizes complementary items and uses category-based similar items to fill the remaining positions.

In [66]:
def get_product_recommendations(
    item_id,
    complementary_n=3,
    top_n=5
):
    complementary_items = [
        recommended_item_id
        for recommended_item_id, score
        in filtered_recommendations.get(item_id, [])
    ]

    similar_items = get_similar_items(item_id, top_n=top_n)

    result = []

    for candidate_id in complementary_items[:complementary_n]:
        if candidate_id != item_id and candidate_id not in result:
            result.append(candidate_id)

    for candidate_id in similar_items:
        if candidate_id != item_id and candidate_id not in result:
            result.append(candidate_id)

        if len(result) == top_n:
            break

    return result

In [68]:
print("Товар із рекомендаціями:", get_product_recommendations(test_item_id))

item_with_category = category_popularity.iloc[0]["itemid"]
print("Товар із категорією:", get_product_recommendations(item_with_category))

print("Невідомий товар:", get_product_recommendations(-1))

Товар із рекомендаціями: [319680, 9877, 208939, 444190, 287449]
Товар із категорією: [283492, 289006, 380388, 220950, 132853]
Невідомий товар: []
